In [1]:
import pandas as pd
from pathlib import Path

In [2]:
project_path = Path("..")
raw_path = project_path / "data" / "raw"
processed_path = project_path / "data" / "processed"

In [4]:
processed_path.mkdir(parents=True, exist_ok= True)

In [7]:
customers = pd.read_csv(raw_path / "customers.csv")
orders = pd.read_csv(raw_path / "orders.csv")

In [11]:
customers_clean = customers.copy()
orders_clean = orders.copy()

ORDERS TABLE

In [16]:
orders_clean.dtypes

order_id                                   str
customer_id                                str
order_date                      datetime64[us]
year                                     int64
month                                    int64
quarter                                    str
day_of_week                                str
product_name                               str
category                                   str
unit_price_usd                         float64
quantity                                 int64
subtotal_usd                           float64
discount_pct                             int64
discount_amount_usd                    float64
shipping_fee_usd                       float64
tax_pct                                  int64
tax_amount_usd                         float64
total_amount_usd                       float64
payment_method                             str
device_used                                str
delivery_days                            int64
delivery_date

In [12]:
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])

In [14]:
orders_clean["delivery_date"] = pd.to_datetime(orders_clean["delivery_date"])

In [17]:
orders_clean["returned"].value_counts(dropna=False)

returned
0    22980
1     2020
Name: count, dtype: int64

In [18]:
orders_clean["is_repeat_customer"].value_counts(dropna=False)

is_repeat_customer
1    16149
0     8851
Name: count, dtype: int64

In [19]:
numeric_cols = [
    "unit_price_usd",
    "quantity",
    "subtotal_usd",
    "discount_pct",
    "discount_amount_usd",
    "shipping_fee_usd",
    "tax_pct",
    "tax_amount_usd",
    "total_amount_usd",
    "delivery_days",
    "customer_rating",
    "session_duration_minutes",
    "pages_viewed_before_purchase"
]

orders_clean[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
unit_price_usd,25000.0,68.124030,57.258933,3.36,29.2500,51.530,87.8775,697.03
quantity,25000.0,1.695520,1.045436,1.00,1.0000,1.000,2.0000,5.00
subtotal_usd,25000.0,116.187322,136.994998,3.36,38.4675,72.390,140.1600,2636.45
discount_pct,25000.0,5.630000,9.740785,0.00,0.0000,0.000,10.0000,50.00
discount_amount_usd,25000.0,6.345500,17.530764,0.00,0.0000,0.000,5.2500,421.58
shipping_fee_usd,25000.0,3.867579,3.269618,0.00,0.0000,3.990,6.9900,9.99
tax_pct,25000.0,10.677080,6.504922,0.00,8.0000,10.000,18.0000,20.00
tax_amount_usd,25000.0,11.746785,17.723056,0.00,2.3000,6.220,13.9900,303.99
total_amount_usd,25000.0,125.456186,145.635016,3.00,43.4500,78.775,149.8725,2730.88
delivery_days,25000.0,4.179480,2.548507,1.00,3.0000,4.000,5.0000,14.00


In [20]:
orders_clean["calculated_subtotal"] = (
    orders_clean["unit_price_usd"] * orders_clean["quantity"]
)

In [21]:
(
    orders_clean["subtotal_usd"].round(2)
    != orders_clean["calculated_subtotal"].round(2)
).sum()

np.int64(0)

In [22]:
orders_clean.drop(columns=["calculated_subtotal"], inplace=True)

In [23]:
calculated_discount = (
    orders_clean["subtotal_usd"]
    * orders_clean["discount_pct"]
    / 100
)

(
    orders_clean["discount_amount_usd"].round(2)
    != calculated_discount.round(2)
).sum()

np.int64(0)

In [24]:
calculated_tax = (
    (orders_clean["subtotal_usd"] - orders_clean["discount_amount_usd"])
    * orders_clean["tax_pct"]
    / 100
)

(
    orders_clean["tax_amount_usd"].round(2)
    != calculated_tax.round(2)
).sum()

np.int64(0)

In [25]:
calculated_total = (
    orders_clean["subtotal_usd"]
    - orders_clean["discount_amount_usd"]
    + orders_clean["shipping_fee_usd"]
    + orders_clean["tax_amount_usd"]
)

(
    orders_clean["total_amount_usd"].round(2)
    != calculated_total.round(2)
).sum()

np.int64(0)

In [26]:
(orders_clean["delivery_date"] < orders_clean["order_date"]).sum()

np.int64(0)

In [27]:
calculated_delivery_days = (
    orders_clean["delivery_date"] - orders_clean["order_date"]
).dt.days

In [ ]:
(
    orders_clean["delivery_days"]
    != calculated_delivery_days
).sum()

np.int64(0)

In [29]:
(
    orders_clean["year"]
    != orders_clean["order_date"].dt.year
).sum()

np.int64(0)

In [30]:
calculated_delivery_days = (
    orders_clean["delivery_date"] - orders_clean["order_date"]
).dt.days

In [31]:
calculated_quarter = (
    "Q" + orders_clean["order_date"].dt.quarter.astype(str)
)

In [32]:
(
    orders_clean["quarter"]
    != calculated_quarter
).sum()

np.int64(0)

In [33]:
calculated_day = orders_clean["order_date"].dt.day_name()

In [ ]:
(
    orders_clean["day_of_week"]
    != calculated_day
).sum()

np.int64(0)

In [36]:
categorical_cols = [
    "quarter",
    "day_of_week",
    "category",
    "payment_method",
    "device_used",
    "order_status",
    "returned",
    "is_repeat_customer"
]

for col in categorical_cols:
    print(orders_clean[col].value_counts(dropna=False))

quarter
Q1    6963
Q4    6157
Q2    5942
Q3    5938
Name: count, dtype: int64
day_of_week
Monday       3658
Sunday       3628
Saturday     3626
Tuesday      3563
Thursday     3560
Friday       3485
Wednesday    3480
Name: count, dtype: int64
category
Electronics               4526
Clothing & Apparel        3981
Home & Kitchen            3068
Books                     1962
Sports & Outdoors         1761
Beauty & Personal Care    1710
Toys & Games              1518
Food & Grocery            1472
Health & Wellness         1219
Jewelry & Accessories      973
Office Supplies            770
Automotive                 768
Pet Supplies               751
Travel & Luggage           521
Name: count, dtype: int64
payment_method
Credit Card             9522
Debit Card              5495
PayPal                  4528
UPI / Digital Wallet    2538
Buy Now Pay Later       1504
Bank Transfer            925
Cryptocurrency           488
Name: count, dtype: int64
device_used
Mobile     13989
Desktop     8090

In [37]:
print("Invalid quantity:", (orders_clean["quantity"] <= 0).sum())

print("Invalid unit price:", (orders_clean["unit_price_usd"] <= 0).sum())

print(
    "Invalid discount:",
    ((orders_clean["discount_pct"] < 0) |
     (orders_clean["discount_pct"] > 100)).sum()
)

print(
    "Invalid tax:",
    ((orders_clean["tax_pct"] < 0) |
     (orders_clean["tax_pct"] > 100)).sum()
)

print(
    "Invalid delivery days:",
    (orders_clean["delivery_days"] < 0).sum()
)

print(
    "Invalid rating:",
    (
        orders_clean["customer_rating"].notna()
        &
        ~orders_clean["customer_rating"].between(1, 5)
    ).sum()
)

Invalid quantity: 0
Invalid unit price: 0
Invalid discount: 0
Invalid tax: 0
Invalid delivery days: 0
Invalid rating: 0


In [38]:
orders_clean["returned"].unique()

array([0, 1])

In [39]:
orders_clean["is_repeat_customer"].unique()

array([1, 0])

In [40]:
orders_clean["returned"] = orders_clean["returned"].astype(bool)
orders_clean["is_repeat_customer"] = orders_clean["is_repeat_customer"].astype(bool)

In [41]:
orders_clean[["returned", "is_repeat_customer"]].dtypes

returned              bool
is_repeat_customer    bool
dtype: object

CUSTOMERS TABLE